In [1]:
import tensorflow as tf
import gc
from tensorflow.keras import backend as K
import os

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Evitar que TensorFlow reserve toda la GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU lista")
print(tf.config.list_physical_devices("GPU"))
def reset_tf():
    K.clear_session()
    gc.collect()

GPU lista
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
import numpy as np
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./famosos/')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

In [3]:
#2. IMPORTAMOS LOS DATOS:
rasgos = pd.read_csv("list_attr_celeba.csv", sep=",")
rasgos = rasgos.head(90000)

photos = []

for file in rasgos["image_id"]:
    #Cargamos la imagen.
    photo = load_img('./famosos/famosos/' + file, target_size=(64, 64))
    #Convertimos la imagen a un array.
    photo = img_to_array(photo)
    #Los guardamos en la lista.
    photos.append(photo)
    del photo

In [4]:
photos = asarray(photos)
rasgos.replace(-1, 0, inplace=True)
rasgos.drop(columns=["image_id"], inplace=True, axis=1)
#rasgos = rasgos[["Attractive", "Bags_Under_Eyes", "Bald"]]
#rasgos = rasgos[["Attractive", "High_Cheekbones", "Mouth_Slightly_Open"]]

photos_reshape = photos.reshape(photos.shape[0],-1) 

In [5]:
from sklearn.model_selection import train_test_split

# Datos originales
X = photos / 255.0
y = rasgos

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Redimensionar las imágenes a un formato de vector
X_train_flat = X_train.reshape(X_train.shape[0], -1)  # Convertir cada imagen a un vector
X_test_flat = X_test.reshape(X_test.shape[0], -1)    # Lo mismo para el conjunto de prueba

In [6]:
print(X_train.shape, y_train.shape)

(72000, 64, 64, 3) (72000, 40)


In [7]:
# Haciendo la red neuronal a partir del tratamiento de PCA
from tensorflow import keras

model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(X_train_flat.shape[1],)))
# Luego metemos capas ocultas
model.add(keras.layers.Dense(256, activation="relu"))
model.add(keras.layers.Dense(128, activation="relu"))
# Luego metemos la capa de salida, que tiene 5 neuronas, una por cada clase, y función de activación softmax, que es la que se suele usar para clasificación multiclase.
model.add(keras.layers.Dense(40, activation="sigmoid"))

from tensorflow.keras import optimizers
sgd = optimizers.Adam(learning_rate=0.00005)

model.compile(loss="binary_crossentropy", optimizer=sgd, metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

2026-04-21 18:45:47.531019: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-21 18:45:47.531231: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-21 18:45:47.531340: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [8]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train_flat, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20


2026-04-21 18:45:51.925487: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-21 18:45:52.981872: I external/local_xla/xla/service/service.cc:168] XLA service 0x728e9e4687f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-21 18:45:52.981898: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-04-21 18:45:53.112455: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-21 18:45:53.376927: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1776789953.582426    7803 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2025/2025 [==============================] - 9s 2ms/step - loss: 0.3709 - accuracy: 0.8440 - val_loss: 0.3352 - val_accuracy: 0.8589
Epoch 2/20
2025/2025 [==============================] - 5s 2ms/step - loss: 0.3225 - accuracy: 0.8629 - val_loss: 0.3144 - val_accuracy: 0.8650
Epoch 3/20
2025/2025 [==============================] - 5s 2ms/step - loss: 0.3078 - accuracy: 0.8680 - val_loss: 0.3036 - val_accuracy: 0.8701
Epoch 4/20
2025/2025 [==============================] - 4s 2ms/step - loss: 0.2992 - accuracy: 0.8712 - val_loss: 0.2977 - val_accuracy: 0.8718
Epoch 5/20
2025/2025 [==============================] - 4s 2ms/step - loss: 0.2936 - accuracy: 0.8734 - val_loss: 0.2932 - val_accuracy: 0.8732
Epoch 6/20
2025/2025 [==============================] - 4s 2ms/step - loss: 0.2891 - accuracy: 0.8751 - val_loss: 0.2923 - val_accuracy: 0.8731
Epoch 7/20
2025/2025 [==============================] - 4s 2ms/step - loss: 0.2852 - accuracy: 0.8765 - val_loss: 0.2895 - val_accuracy: 0.8736
Epo

In [9]:
model.evaluate(X_test_flat, y_test)

563/563 [==============================] - 1s 1ms/step - loss: 0.2718 - accuracy: 0.8818


[0.27183201909065247, 0.8818264007568359]

In [10]:
del model
reset_tf()

## Vamos a hacer la red neuronal convolucional

In [11]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(64, 64, 3)))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(512, activation="relu"))
model_cnn.add(keras.layers.Dense(256, activation="relu"))
model_cnn.add(keras.layers.Dense(40, activation="sigmoid"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.00001)
model_cnn.compile(optimizer=sgd_cnn, loss="binary_crossentropy", metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

In [12]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20


2026-04-21 18:47:27.928247: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


2025/2025 [==============================] - 15s 6ms/step - loss: 0.3843 - accuracy: 0.8412 - val_loss: 0.3389 - val_accuracy: 0.8590
Epoch 2/20
2025/2025 [==============================] - 11s 6ms/step - loss: 0.3218 - accuracy: 0.8646 - val_loss: 0.3113 - val_accuracy: 0.8679
Epoch 3/20
2025/2025 [==============================] - 11s 6ms/step - loss: 0.3008 - accuracy: 0.8719 - val_loss: 0.2961 - val_accuracy: 0.8733
Epoch 4/20
2025/2025 [==============================] - 11s 6ms/step - loss: 0.2886 - accuracy: 0.8764 - val_loss: 0.2876 - val_accuracy: 0.8763
Epoch 5/20
2025/2025 [==============================] - 11s 6ms/step - loss: 0.2807 - accuracy: 0.8792 - val_loss: 0.2801 - val_accuracy: 0.8789
Epoch 6/20
2025/2025 [==============================] - 11s 6ms/step - loss: 0.2748 - accuracy: 0.8813 - val_loss: 0.2752 - val_accuracy: 0.8808
Epoch 7/20
2025/2025 [==============================] - 12s 6ms/step - loss: 0.2703 - accuracy: 0.8831 - val_loss: 0.2720 - val_accuracy: 0.8

In [13]:
model_cnn.evaluate(X_test, y_test)

563/563 [==============================] - 1s 2ms/step - loss: 0.2529 - accuracy: 0.8896


[0.25292256474494934, 0.8895763754844666]

In [14]:
del model_cnn
reset_tf()